# **Assignment 1**

## **Solving a linear system in python**

### **1. Parsing the System of Equations (1 point)**

In [1]:
import pathlib
import tempfile


def load_system(path: pathlib.Path) -> tuple[list[list[float]], list[float]]:
    matrix, vector = [], []
    for line in path.read_text(encoding="utf-8-sig").splitlines():
        if not line.strip():
            continue
        left, right = line.split("=")
        terms = left.split()
        if len(terms) != 5:
            raise ValueError("Expected three terms in x, y, z order.")
        row = []
        for index, variable in enumerate("xyz"):
            term = terms[2 * index]
            if not term.endswith(variable):
                raise ValueError("Expected variables in x, y, z order.")
            coefficient = term[:-1]
            if coefficient in ("", "+"):
                value = 1.0
            elif coefficient == "-":
                value = -1.0
            else:
                value = float(coefficient)
            if index > 0:
                operator = terms[2 * index - 1]
                if operator not in ("+", "-"):
                    raise ValueError("Expected + or - between terms.")
                if operator == "-":
                    value = -value
            row.append(value)
        matrix.append(row)
        vector.append(float(right.strip()))
    if len(matrix) != 3:
        raise ValueError("Expected exactly three equations.")
    return matrix, vector


with tempfile.TemporaryDirectory() as directory:
    system_path = pathlib.Path(directory) / "system.txt"
    system_path.write_text(
        "2x + 3y - z = 5\nx - y + 4z = 6\n3x + y + 2z = 7\n",
        encoding="utf-8",
    )
    A, B = load_system(system_path)

print(f"{A=} {B=}")


A=[[2.0, 3.0, -1.0], [1.0, -1.0, 4.0], [3.0, 1.0, 2.0]] B=[5.0, 6.0, 7.0]


### **2. Matrix and Vector Operations (5 points)**

#### 2.1. Determinant

In [2]:
def determinant(matrix: list[list[float]]) -> float:
    size = len(matrix)
    if size not in (1, 2, 3) or any(len(row) != size for row in matrix):
        raise ValueError("Expected a square matrix of size 1, 2, or 3.")
    if size == 1:
        return matrix[0][0]
    if size == 2:
        return matrix[0][0] * matrix[1][1] - matrix[0][1] * matrix[1][0]
    a, b, c = matrix[0]
    d, e, f = matrix[1]
    g, h, i = matrix[2]
    return a * (e * i - f * h) - b * (d * i - f * g) + c * (d * h - e * g)


print(f"{determinant(A)=}")


determinant(A)=14.0


#### 2.2. Trace

In [3]:
def trace(matrix: list[list[float]]) -> float:
    if any(len(row) != len(matrix) for row in matrix):
        raise ValueError("Expected a square matrix.")
    return sum(matrix[i][i] for i in range(len(matrix)))


print(f"{trace(A)=}")


trace(A)=3.0


#### 2.3. Vector norm

In [4]:
def norm(vector: list[float]) -> float:
    return sum(value ** 2 for value in vector) ** 0.5


print(f"{norm(B)=}")


norm(B)=10.488088481701515


#### 2.4. Transpose of matrix

In [5]:
def transpose(matrix: list[list[float]]) -> list[list[float]]:
    if not matrix:
        return []
    columns = len(matrix[0])
    if any(len(row) != columns for row in matrix):
        raise ValueError("All rows must have the same length.")
    return [[matrix[i][j] for i in range(len(matrix))] for j in range(columns)]


print(f"{transpose(A)=}")


transpose(A)=[[2.0, 1.0, 3.0], [3.0, -1.0, 1.0], [-1.0, 4.0, 2.0]]


#### 2.5. Matrix-vector multiplication

In [6]:
def multiply(matrix: list[list[float]], vector: list[float]) -> list[float]:
    if any(len(row) != len(vector) for row in matrix):
        raise ValueError("Matrix columns must match the vector length.")
    return [sum(row[j] * vector[j] for j in range(len(vector))) for row in matrix]


print(f"{multiply(A, B)=}")


multiply(A, B)=[21.0, 27.0, 35.0]


### **3. Solving using Cramer's Rule (1 point)**

In [7]:
def solve_cramer(matrix: list[list[float]], vector: list[float]) -> list[float]:
    if len(matrix) != 3 or any(len(row) != 3 for row in matrix) or len(vector) != 3:
        raise ValueError("Expected a 3 by 3 matrix and a vector of length 3.")
    det = determinant(matrix)
    if det == 0:
        raise ValueError("The system does not have a unique solution.")
    solution = []
    for column in range(3):
        replaced = [row[:] for row in matrix]
        for row in range(3):
            replaced[row][column] = vector[row]
        solution.append(determinant(replaced) / det)
    return solution


print(f"{solve_cramer(A, B)=}")


solve_cramer(A, B)=[0.35714285714285715, 2.0714285714285716, 1.9285714285714286]


### **4. Solving using Inversion (3 points)**

In [8]:
def minor(matrix: list[list[float]], i: int, j: int) -> list[list[float]]:
    return [
        [value for column, value in enumerate(row) if column != j]
        for index, row in enumerate(matrix)
        if index != i
    ]


def cofactor(matrix: list[list[float]]) -> list[list[float]]:
    if len(matrix) != 3 or any(len(row) != 3 for row in matrix):
        raise ValueError("Expected a 3 by 3 matrix.")
    return [
        [(-1) ** (i + j) * determinant(minor(matrix, i, j)) for j in range(3)]
        for i in range(3)
    ]


def adjoint(matrix: list[list[float]]) -> list[list[float]]:
    return transpose(cofactor(matrix))


def solve(matrix: list[list[float]], vector: list[float]) -> list[float]:
    if len(matrix) != 3 or any(len(row) != 3 for row in matrix) or len(vector) != 3:
        raise ValueError("Expected a 3 by 3 matrix and a vector of length 3.")
    det = determinant(matrix)
    if det == 0:
        raise ValueError("The system does not have a unique solution.")
    inverse = [[value / det for value in row] for row in adjoint(matrix)]
    return multiply(inverse, vector)


print(f"{solve(A, B)=}")


solve(A, B)=[0.3571428571428572, 2.0714285714285716, 1.9285714285714284]


### Additional systems

In [9]:
systems = [
    "x + y + z = 6\n2x - y + 3z = 9\n-x + 4y + z = 10\n",
    "0x + 2y - z = 7\nx - y + z = -2\n2x + y + z = 3\n",
    "-x + 2.5y - 0.5z = -4.5\n3x + y + 2z = 8\n2x - 3y + z = 10\n",
]

with tempfile.TemporaryDirectory() as directory:
    for index, equations in enumerate(systems, start=1):
        path = pathlib.Path(directory) / f"system_{index}.txt"
        path.write_text(equations, encoding="utf-8")
        matrix, vector = load_system(path)
        print(f"System {index}: A={matrix}, B={vector}")
        print(f"Cramer's rule: {solve_cramer(matrix, vector)}")
        print(f"Inversion: {solve(matrix, vector)}")


System 1: A=[[1.0, 1.0, 1.0], [2.0, -1.0, 3.0], [-1.0, 4.0, 1.0]], B=[6.0, 9.0, 10.0]
Cramer's rule: [1.0, 2.0, 3.0]
Inversion: [1.0000000000000009, 1.9999999999999998, 2.9999999999999996]
System 2: A=[[0.0, 2.0, -1.0], [1.0, -1.0, 1.0], [2.0, 1.0, 1.0]], B=[7.0, -2.0, 3.0]
Cramer's rule: [5.0, -0.0, -7.0]
Inversion: [5.0, 0.0, -7.0]
System 3: A=[[-1.0, 2.5, -0.5], [3.0, 1.0, 2.0], [2.0, -3.0, 1.0]], B=[-4.5, 8.0, 10.0]
Cramer's rule: [15.5, 0.5, -19.5]
Inversion: [15.5, 0.5, -19.5]
